In [ ]:
### IMPORTS AND HELPER FUNCTIONS
import json

import matplotlib.pyplot as plt

from dynamics_analysis_with_koopman.config.configs import PROJECT_ROOT
from dynamics_analysis_with_koopman.utils.checkpoint_registry import (
    checkpoint_path_for,
    load_train_tsc_from_checkpoint,
    load_run_from_checkpoint,
)
from dynamics_analysis_with_koopman.plots import (
    plot_modes,
    generate_modes_gif,
    generate_eigenfunction_modes_gif,
    plot_supermodes,
)

import warnings

warnings.filterwarnings('ignore')

FIGURE_FORMAT = 'svg'
GIF_FILETYPE = 'gif'

In [ ]:
# load models/data from the run registry, keyed by experiment name -- falls back
# to a literal run id for checkpoints the registry does not track
RUN_REGISTRY = PROJECT_ROOT / 'papers' / 'clustering' / 'juelich_run_registry.json'

with open(RUN_REGISTRY) as f:
    registry = json.load(f)

# all plots/gifs from this notebook are redirected here, instead of the checkpoint directory
MODES_ROOT = PROJECT_ROOT / 'papers' / 'clustering' / 'modes'


def load_model_with_config(experiment=None, run_id=None, model_path='edmd_model'):
    edmd, run_config, run_metrics, style = load_run_from_checkpoint(
        experiment=experiment,
        run_id=run_id,
        registry=registry,
        model_name=model_path,
    )
    print(f'Found run at {style.path}')
    print('Model loaded successfully.')
    return edmd, run_config, run_metrics, style


def load_training_data(experiment=None, run_id=None):
    checkpoint_path = checkpoint_path_for(experiment=experiment, run_id=run_id, registry=registry)
    print(f'Found run at {checkpoint_path}')

    # get the eigenfunction values for the training data
    try:
        train_tsc = load_train_tsc_from_checkpoint(checkpoint_path)
        print('Train data loaded successfully.')

    except FileNotFoundError:
        print(
            f'Could not load the training data tsc, which is needed to retrieve the eigenfunction values per series and time step.'
        )

    return train_tsc


def get_eigenfunction_values(edmd, series_ids, experiment=None, run_id=None):
    train_tsc = load_training_data(experiment=experiment, run_id=run_id)
    print(f'Applying eigenfunctions to series ids {series_ids}')
    eval_eigenfunctions = edmd.koopman_eigenfunction(train_tsc.loc[series_ids])  # returns TSCDataFrame
    print(f'Done!')

    return eval_eigenfunctions, train_tsc

In [ ]:
experiment = '2009unidirOpenBs'  # 2009unidirOpenBs_takens_dmd_checkpoint

MEDIUM_SIZE = 16
CBAR_WIDTH = 0.008  # narrower than the plots library default (0.015)

plt.rc('legend', fontsize=MEDIUM_SIZE)  # legend fontsize
plt.rc('axes', labelsize=MEDIUM_SIZE)
plt.rc('xtick', labelsize=MEDIUM_SIZE)
plt.rc('ytick', labelsize=MEDIUM_SIZE)

# get the model and configs
edmd, run_config, metrics, style = load_model_with_config(experiment=experiment)
style = style.with_(path=MODES_ROOT / experiment, fig_filetype=FIGURE_FORMAT, dpi=300)

mode_selection = [1, 2]
VECTOR_SCALING = 0.004

for observable_type in ['density', 'velocity']:
    (selection, modes, eigenvalues, _, _) = plot_modes(
        model=edmd,
        tsc=None,
        observable_type=observable_type,
        style=style.with_(vector_scaling=VECTOR_SCALING, koopman_selection=mode_selection),
        cbar_width=CBAR_WIDTH,
    )

In [ ]:
experiment = '2013unidirectionalB'

MEDIUM_SIZE = 20
CBAR_WIDTH = 0.008  # narrower than the plots library default (0.015)

plt.rc('legend', fontsize=MEDIUM_SIZE)  # legend fontsize
plt.rc('axes', labelsize=MEDIUM_SIZE)
plt.rc('xtick', labelsize=MEDIUM_SIZE)
plt.rc('ytick', labelsize=MEDIUM_SIZE)

# get the model and configs
edmd, run_config, metrics, style = load_model_with_config(experiment=experiment)
style = style.with_(path=MODES_ROOT / experiment, fig_filetype=FIGURE_FORMAT, dpi=300)

mode_selection = [1, 2]
VECTOR_SCALING = 0.004

for observable_type in ['density', 'velocity']:
    (selection, modes, eigenvalues, _, _) = plot_modes(
        model=edmd,
        tsc=None,
        observable_type=observable_type,
        style=style.with_(vector_scaling=VECTOR_SCALING, koopman_selection=mode_selection),
        cbar_width=CBAR_WIDTH,
    )

In [ ]:
# plot modes for lane formation example (matched with 2013bidirectionalB)
experiment = '2013crossing120A'

# get the model and configs
edmd, run_config, metrics, style = load_model_with_config(experiment=experiment)
style = style.with_(path=MODES_ROOT / experiment, fig_filetype=FIGURE_FORMAT, dpi=300)

observable_type = 'velocity'
VECTOR_SCALING = 0.01
mode_selection = [11]

train_tsc = load_training_data(experiment=experiment)

# series_ids = [10012010101, 10012010202, 10012010303, 10012010404, 10012010505, 10012010606, 10012010707]
series_ids = [10012010505]
eigenfunctions_tsc, train_tsc = get_eigenfunction_values(edmd, series_ids, experiment=experiment)

# length of the gif is based on the series length
generate_eigenfunction_modes_gif(
    model=edmd,
    eigenfunctions_tsc=eigenfunctions_tsc,
    selection=mode_selection,
    series_ids=series_ids,
    observable_type=observable_type,
    style=style.with_(vector_scaling=VECTOR_SCALING, gif_filetype=GIF_FILETYPE),
)

VECTOR_SCALING = 0.005
# normalized modes gif for the appendix
generate_modes_gif(
    model=edmd,
    selection=mode_selection,
    observable_type=observable_type,
    tmax=int(62 / 0.2),  # time_values are exactly the steps we predict
    style=style.with_(vector_scaling=VECTOR_SCALING, gif_filetype=GIF_FILETYPE),
    normalize_ev=True,
)

In [ ]:
experiment = '2013crossing90D'
edmd, run_config, metrics, style = load_model_with_config(experiment=experiment)
style = style.with_(path=MODES_ROOT / experiment, fig_filetype=FIGURE_FORMAT, gif_filetype=GIF_FILETYPE)

MEDIUM_SIZE = 25
CBAR_WIDTH = 0.008  # narrower than the plots library default (0.015)

plt.rc('legend', fontsize=MEDIUM_SIZE)  # legend fontsize
plt.rc('axes', labelsize=MEDIUM_SIZE)
plt.rc('xtick', labelsize=MEDIUM_SIZE)
plt.rc('ytick', labelsize=MEDIUM_SIZE)

observable_type = 'density'
# VECTOR_SCALING = 0.02

series_ids = [10090411, 10090422, 10090433, 10090455, 10090466, 10090477, 10090488]
eigenfunctions_tsc, train_tsc = get_eigenfunction_values(edmd, series_ids, experiment=experiment)

# print the mean, max and min of the absolute value per series for eigenfunction 28
eigenfunction_idx = 28
abs_eigenfunction = eigenfunctions_tsc[f'koop_eigfunc{eigenfunction_idx - 1}'].abs()
print(f'|eigenfunction {eigenfunction_idx}| per series:')
print(abs_eigenfunction.groupby(level=0).agg(['mean', 'median', 'max', 'min']))

# for the density contribution of the first mode only
plot_supermodes(
    model=edmd,
    tsc=train_tsc,
    observable_type=observable_type,
    style=style.with_(koopman_selection=[1], visualization_timesteps=[int(18 / 0.2)]),
)

MEDIUM_SIZE = 16
plt.rc('legend', fontsize=MEDIUM_SIZE)  # legend fontsize
plt.rc('axes', labelsize=MEDIUM_SIZE)
plt.rc('xtick', labelsize=MEDIUM_SIZE)
plt.rc('ytick', labelsize=MEDIUM_SIZE)

modes_gifs = [28]
# "abs" for black and white coloring for main part to accentuate shapes
generate_modes_gif(
    model=edmd,
    selection=modes_gifs,
    observable_type=observable_type,
    tmax=int(14 / 0.2),  # time_values are exactly the steps we predict
    style=style.with_(contour=True, abs_value=True, color_levels=10),
    normalize_ev=True,
)

# full color palatte for the appendix
generate_modes_gif(
    model=edmd,
    selection=modes_gifs,
    observable_type=observable_type,
    tmax=int(9 / 0.2),  # time_values are exactly the steps we predict
    style=style,  # .with_(contour=True),
    normalize_ev=True,
)

In [ ]:
experiment = '2013crossing90E'
edmd, run_config, metrics, style = load_model_with_config(experiment=experiment)
style = style.with_(path=MODES_ROOT / experiment, fig_filetype=FIGURE_FORMAT, gif_filetype=GIF_FILETYPE)

observable_type = 'density'

MEDIUM_SIZE = 25
CBAR_WIDTH = 0.008  # narrower than the plots library default (0.015)

plt.rc('legend', fontsize=MEDIUM_SIZE)  # legend fontsize
plt.rc('axes', labelsize=MEDIUM_SIZE)
plt.rc('xtick', labelsize=MEDIUM_SIZE)
plt.rc('ytick', labelsize=MEDIUM_SIZE)

series_ids = [10090511, 10090566]
eigenfunctions_tsc, train_tsc = get_eigenfunction_values(edmd, series_ids, experiment=experiment)

# print the mean, max and min of the absolute value per series for eigenfunction 28
eigenfunction_idx = 10
abs_eigenfunction = eigenfunctions_tsc[f'koop_eigfunc{eigenfunction_idx - 1}'].abs()
print(f'|eigenfunction {eigenfunction_idx}| per series:')
print(abs_eigenfunction.groupby(level=0).agg(['mean', 'median', 'max', 'min']))

# for the density contribution of the first mode only
plot_supermodes(
    model=edmd,
    tsc=train_tsc,
    observable_type=observable_type,
    style=style.with_(koopman_selection=[1], visualization_timesteps=[int(18 / 0.2)]),
)

MEDIUM_SIZE = 16
plt.rc('legend', fontsize=MEDIUM_SIZE)  # legend fontsize
plt.rc('axes', labelsize=MEDIUM_SIZE)
plt.rc('xtick', labelsize=MEDIUM_SIZE)
plt.rc('ytick', labelsize=MEDIUM_SIZE)

modes_gifs = [10] 
# "abs" for black and white coloring for main part to accentuate shapes
generate_modes_gif(
    model=edmd,
    selection=modes_gifs,
    observable_type=observable_type,
    tmax=int(14 / 0.2),  # time_values are exactly the steps we predict
    style=style.with_(contour=True, abs_value=True, color_levels=10),
    normalize_ev=True,
)

# full color palatte for the appendix
generate_modes_gif(
    model=edmd,
    selection=modes_gifs,
    observable_type=observable_type,
    tmax=int(9 / 0.2),  # time_values are exactly the steps we predict
    style=style,  # .with_(contour=True),
    normalize_ev=True,
)

In [ ]:
# plot modes for lane formation example (matched with 2013crossing120A)
experiment = '2013bidirectionalB'

# get the model and configs
edmd, run_config, metrics, style = load_model_with_config(experiment=experiment, model_path='edmd_model')
style = style.with_(path=MODES_ROOT / experiment, fig_filetype=FIGURE_FORMAT, dpi=300)

observable_type = 'velocity'
VECTOR_SCALING = 0.01
mode_selection = [9]

# series_ids = [1004000101, 1004000202, 1004000303, 1004000404, 1004000505, 1004000606, 1004000707, 1004000808, 1004000909, 1004001010,]
series_ids = [1004000505]
eigenfunctions_tsc, train_tsc = get_eigenfunction_values(edmd, series_ids, experiment=experiment)

# length of the gif is based on the series length
generate_eigenfunction_modes_gif(
    model=edmd,
    eigenfunctions_tsc=eigenfunctions_tsc,
    selection=mode_selection,
    series_ids=series_ids,
    observable_type=observable_type,
    style=style.with_(vector_scaling=VECTOR_SCALING, gif_filetype=GIF_FILETYPE),
)

VECTOR_SCALING = 0.005
# normalized modes gif for the appendix
generate_modes_gif(
    model=edmd,
    selection=mode_selection,
    observable_type=observable_type,
    tmax=int(62 / 0.2),  # time_values are exactly the steps we predict
    style=style.with_(vector_scaling=VECTOR_SCALING, gif_filetype=GIF_FILETYPE),
    normalize_ev=True,
)

In [ ]:
## for method overview
experiment = '2017simeB'  # 2017simeB_takens_dmd_checkpoint

# get the model and configs
edmd, run_config, metrics, style = load_model_with_config(experiment=experiment, model_path='edmd_model')
style = style.with_(path=MODES_ROOT / experiment, fig_filetype=FIGURE_FORMAT, dpi=300)

observable_type = 'density'
mode_selection = [3, 4, 1, 2]
VECTOR_SCALING = 0.004

# plot the modes for method overview
(selection, modes, eigenvalues, _, _) = plot_modes(
    model=edmd,
    tsc=None,
    observable_type=observable_type,
    style=style.with_(vector_scaling=VECTOR_SCALING, koopman_selection=mode_selection),
    cbar_width=CBAR_WIDTH,
)

In [ ]:
experiment = '2006bottleneck'  # 2006bottleneck_takens_dmd_checkpoint

observable_type = 'density'

SMALL_SIZE = 18
MEDIUM_SIZE = 24
CBAR_WIDTH = 0.008  # narrower than the plots library default (0.015)

plt.rc('legend', fontsize=MEDIUM_SIZE)  # legend fontsize
plt.rc('axes', labelsize=MEDIUM_SIZE, titlesize=MEDIUM_SIZE)
plt.rc('xtick', labelsize=SMALL_SIZE)
plt.rc('ytick', labelsize=SMALL_SIZE)
plt.rc('figure', labelsize=SMALL_SIZE, titlesize=MEDIUM_SIZE)

edmd, run_config, metrics, style = load_model_with_config(experiment=experiment, model_path='edmd_model')
style = style.with_(path=MODES_ROOT / experiment, fig_filetype=FIGURE_FORMAT, dpi=300)

mode_selection = [1, 2, 3, 4, 5, 6, 7]

(selection, modes, eigenvalues, _, _) = plot_modes(
    model=edmd,
    tsc=None,
    observable_type=observable_type,
    style=style.with_(koopman_selection=mode_selection),
)

train_tsc = load_training_data(experiment=experiment)

plot_supermodes(
    model=edmd,
    tsc=train_tsc,
    observable_type=observable_type,
    style=style.with_(
        koopman_selection=mode_selection, visualization_timesteps=[10]
    ),  # make steps of t = 65 for 13s (phase of 2*13=26s)
    vmax=1,
    # vmin=-.5,
)